# Fine-tune Llama 3.2 (3B)


## 1. Install

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps "trl<0.16.0" peft accelerate bitsandbytes


## 2. Load Llama 3.2 3B (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length, dtype = None, load_in_4bit = True,
)


## 3. Upload `music_prompts.jsonl`

In [ ]:
from google.colab import files
uploaded = files.upload()
print("uploaded:", list(uploaded.keys()))


Saving music_prompts.jsonl to music_prompts.jsonl
uploaded: ['music_prompts.jsonl']


## 4. Add LoRA adapters (rank 16)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407,
)


Unsloth 2026.6.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 5. Format with the chat template

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
def fmt(ex):
    return {"text":[tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
                    for c in ex["messages"]]}
dataset = load_dataset("json", data_files="music_prompts.jsonl", split="train").map(fmt, batched=True)
print(dataset, "\n\n", dataset[0]["text"][:500])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'text'],
    num_rows: 2500
}) 

 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You convert a natural-language music request into a single valid JSON object with exactly these keys: genre, mood, tempo_bpm, intensity, vocals, instruments, context, tags. Infer only what the request reasonably implies; do not invent a genre or vocals the request does not support. For vague requests, choose a common, safe genre. Respond with JSON only.<|eot_id|><|start_he


In [ ]:
split = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset=split["train"]
val_dataset=split["test"]

## 6. Trainer + response-only masking

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field="text", max_seq_length=max_seq_length, packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=10,
        num_train_epochs=3, learning_rate=2e-4, logging_steps=20, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="linear", seed=3407,
        output_dir="outputs", report_to="none",
    ),

    # train_dataset=train_dataset,
    # eval_dataset=val_dataset,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=6):   0%|          | 0/2500 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/2500 [00:00<?, ? examples/s]

## 7. Train  (~30–60 min on a free T4)

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,500 | Num Epochs = 3 | Total steps = 939
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
20,1.096376
40,0.466741
60,0.352444
80,0.289876
100,0.260779
120,0.240536
140,0.230643
160,0.222475
180,0.214029
200,0.216388


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-939/tokenizer_config.json.


Step,Training Loss
20,1.096376
40,0.466741
60,0.352444
80,0.289876
100,0.260779
120,0.240536
140,0.230643
160,0.222475
180,0.214029
200,0.216388


## 8. Generate → validate (8-key schema) → render audio prompt

In [ ]:
import json, re
SYSTEM = """You convert a natural-language music request into a single valid JSON object with exactly these keys: genre, mood, tempo_bpm, intensity, vocals, instruments, context, tags. Infer only what the request reasonably implies; do not invent a genre or vocals the request does not support. For vague requests, choose a common, safe genre. Respond with JSON only."""

VOCALS={"none","male","female","choir","vocal-chops","spoken"}
REQUIRED={"genre","mood","tempo_bpm","intensity","vocals","instruments","context","tags"}
def extract_json(t):
    m=re.search(r"```(?:json)?\s*(\{.*?\})\s*```",t,re.DOTALL)
    if m: t=m.group(1)
    s,e=t.find("{"),t.rfind("}"); return json.loads(t[s:e+1])
def validate(o):
    er=[]
    if REQUIRED-set(o): er.append(f"missing {sorted(REQUIRED-set(o))}")
    if "tempo_bpm" in o and not (isinstance(o["tempo_bpm"],int) and 0<=o["tempo_bpm"]<=300): er.append("bad tempo_bpm")
    if "intensity" in o and not (isinstance(o["intensity"],(int,float)) and 0<=o["intensity"]<=1): er.append("bad intensity")
    if o.get("vocals") not in VOCALS: er.append("bad vocals")
    return (not er),er

def render(o):
    bpm=o.get("tempo_bpm",0); voc=o.get("vocals","none")
    tempo=("beatless" if not bpm else "slow" if bpm<70 else "mid-tempo" if bpm<120 else "driving")
    instr=", ".join(o.get("instruments",[])[:5]); mood=", ".join(o.get("mood",[])[:3])
    v="fully instrumental" if voc=="none" else f"with {voc} vocals"
    c=o.get("context",{}); act=c.get("activity"); env=c.get("environment")
    scene=(f", for {act}" if act else "")+(f" in {env}" if env and env!="any" else "")
    return f"A {tempo} {o.get('genre','music')} track{' around '+str(bpm)+' BPM' if bpm else ''}, "\
           f"featuring {instr}, {v}, {mood} in feel{scene}."

FastLanguageModel.for_inference(model)
def run(prompt, temperature=0.3):
    msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}]
    ids=tokenizer.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors="pt").to("cuda")
    out=model.generate(input_ids=ids,max_new_tokens=400,temperature=temperature,do_sample=True,
                       pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:],skip_special_tokens=True)

tests=["lofi to study to while it is raining at night","aggressive gym trap, hard hitting",
       "calm acoustic morning coffee","epic orchestral battle scene","cool music for a beach at sunset"]
ok=0
for p in tests:
    raw=run(p)
    try: o=extract_json(raw); v,er=validate(o)
    except Exception as ex: v,er,o=False,[f"parse fail: {ex}"],None
    ok+=v; print(f"[{'OK ' if v else 'BAD'}] {p}")
    if v: print("     ",o["genre"],"| bpm",o["tempo_bpm"],"| vocals",o["vocals"]); print("      ->",render(o))
    else: print("     ",er,"|",raw[:140])
print(f"\nvalid: {ok}/{len(tests)}")


## 9. Save & download the adapter

In [ ]:
model.save_pretrained("music_lora"); tokenizer.save_pretrained("music_lora")
!zip -r music_lora.zip music_lora >/dev/null
from google.colab import files; files.download("music_lora.zip")

Unsloth: Restored added_tokens_decoder metadata in music_lora/tokenizer_config.json.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
**If a cell errors** (libraries drift): open Unsloth's official Llama 3.2 (3B) Colab
(linked from `huggingface.co/unsloth/Llama-3.2-3B-Instruct`), run it once unchanged, then
swap in cells 3 and 5 above. **Inference later:** reload `music_lora`, call `run(prompt)`,
pass the output through `validate()` (retry once if BAD), then `render()` before sending to
your audio model. The standalone `render.py` file is the fuller version of the render step.
